# Analyze bulk RNA-seq

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/publications/intestine-tf")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import numpy as np
import pandas as pd
import seaborn as sns
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.ticker import LogLocator, LogFormatterMathtext
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from scipy.stats import ttest_ind

plt.style.use('default')


# ---- Scanpy global settings ----
sc.settings.verbosity = 2
sc.settings.autoshow = False
sc.settings.set_figure_params(
    dpi=150,
    dpi_save=300,
    format="pdf",
    facecolor="white",
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(4, 4),
    transparent=True,
)

# ---- Matplotlib global settings ----
mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.fontsize": 6,
        "axes.titlesize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)


mpl.rcParams.update({
    "svg.fonttype": "none",       # Keep SVG text selectable
    "pdf.fonttype": 42,           # Embed fonts in PDFs (TrueType)
})

## Reformat for pipeline

In [ ]:
# Set base directory
base_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/bulkrna")

# Set data directory
data_dir = base_dir / "raw" 

# Set output directory
output_dir = base_dir / "processed"
output_dir.mkdir(exist_ok=True)

In [ ]:
# Load metadata df
metadata_df = pd.read_csv(data_dir / "biokitr" / "metadata.tsv", sep="\t")

# Rename sample_id to sample
metadata_df.rename(columns={"sample_id": "sample"}, inplace=True)

metadata_df["condition"] = metadata_df["sample"].str.split("_").str[0]
metadata_df["replicate"] = "Rep" + metadata_df["sample"].str.split("_").str[1]

In [ ]:
# Create samplesheet with sample, condition, replicate columns
samplesheet_df = metadata_df[["sample", "condition", "replicate"]]

# Write samplesheet to file
samplesheet_df.to_csv(output_dir / "samplesheet.csv", index=False)

In [ ]:
# Specify contrasts df with columns id, variable, reference, target
contrasts_df = pd.DataFrame(
    {
        "id": ["MS_vs_Control", "PD_vs_Control"],
        "variable": ["condition", "condition"],
        "reference": ["Control", "Control"],
        "target": ["MS023", "PD173074"]
    }
)

# Write contrasts to file
contrasts_df.to_csv(output_dir / "contrasts.csv", index=False)

In [ ]:
# Load counts matrix
counts_df = pd.read_csv(data_dir / "biokitr" / "counts.gct", sep="\t", skiprows=2)

In [ ]:
# Rename Description to gene_id
counts_df.rename(columns={"Description": "gene_id"}, inplace=True)

In [ ]:
counts_df.head()

In [ ]:
# Write matrix counts
counts_df.to_csv(output_dir / "counts.txt", sep="\t", index=False)

## Load the processed data

In [ ]:
# Load the annotated normalised counts from DESeq2
counts_df = pd.read_csv(output_dir / "deseq2" / "tables" / "processed_abundance" / "annotated.all.normalised_counts.tsv", sep="\t")
samplesheet_df = pd.read_csv(output_dir / "samplesheet.csv")
ms_vs_control_de_df = pd.read_csv(output_dir / "deseq2" / "tables" / "differential" / "annotated.MS_vs_Control.deseq2.results.tsv", sep="\t").assign(comparison="MS_vs_Control")
pd_vs_control_de_df = pd.read_csv(output_dir / "deseq2" / "tables" / "differential" / "annotated.PD_vs_Control.deseq2.results.tsv", sep="\t").assign(comparison="PD_vs_Control")
de_df = pd.concat([ms_vs_control_de_df, pd_vs_control_de_df], ignore_index=True)

In [ ]:
order = ["Control", "MS023", "PD173074"]

def p_to_stars(p):
    return "****" if p < 1e-4 else "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "ns"

def add_sig_bar(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.2, c="black")
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=10)

for gene in ["NEUROG3", "GCG", "TPH1", "GHRL", "MLN", "GIP", "SST", "CCK", "PYY"]:
    #  Subset to gene
    gene_df = counts_df.loc[counts_df["gene_name"] == gene]

    # Add condition and replicate information
    gene_df = gene_df.melt(id_vars=["gene_name"], var_name="sample", value_name="normalized_count")
    gene_df = gene_df.merge(samplesheet_df[["sample", "condition", "replicate"]], on="sample")

    # Check that the gene is present in the DESeq2 results before trying to access it
    if gene not in de_df["gene_name"].values:
        print(f"{gene} not found in DESeq2 results, skipping.")
        continue

    # Extract the testing p-values from the DESeq2 results
    p_ms023 = de_df.query(f"gene_name == '{gene}' & comparison == 'MS_vs_Control'")["padj"].values[0]
    log2fc_ms023 = de_df.query(f"gene_name == '{gene}' & comparison == 'MS_vs_Control'")["log2FoldChange"].values[0]
    p_pd173074 = de_df.query(f"gene_name == '{gene}' & comparison == 'PD_vs_Control'")["padj"].values[0]
    log2fc_pd173074 = de_df.query(f"gene_name == '{gene}' & comparison == 'PD_vs_Control'")["log2FoldChange"].values[0]

    print(f"{gene} - MS023 vs Control p-value: {p_ms023:.4g}, PD173074 vs Control p-value: {p_pd173074:.4g}")
    print(f"{gene} - MS023 vs Control log2FC: {log2fc_ms023:.4f}, PD173074 vs Control log2FC: {log2fc_pd173074:.4f}")

    with plt.rc_context({"axes.grid": False}):
        fig, ax = plt.subplots(figsize=(3.2, 3.8))

        sns.barplot(
            data=gene_df,
            x="condition",
            y="normalized_count",
            order=order,
            errorbar="sd",
            capsize=0.15,
            color="lightgrey",
            err_kws={"linewidth": 1.2},
            edgecolor="black",
            linewidth=1,
            ax=ax,
        )

        sns.stripplot(
            data=gene_df,
            x="condition",
            y="normalized_count",
            order=order,
            color="black",
            size=5,
            jitter=0.15,
            alpha=0.8,
            ax=ax,
        )

        ymax = gene_df["normalized_count"].max()
        yrange = gene_df["normalized_count"].max() - gene_df["normalized_count"].min()
        print(gene_df)
        h = yrange * 0.05

        add_sig_bar(ax, 0, 1, ymax + yrange * 0.10, h, p_to_stars(p_ms023))
        add_sig_bar(ax, 0, 2, ymax + yrange * 0.27, h, p_to_stars(p_pd173074))

        ax.set_title(f"{gene_df['gene_name'].iloc[0]} expression", fontsize=12)
        ax.set_xlabel("")
        ax.set_ylabel("Normalized count")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        ax.set_yscale("log")

        # optional: remove minor ticks
        ax.set_ylim(bottom=1, top=ymax + yrange * 0.45)

        ax.yaxis.set_major_locator(LogLocator(base=10))
        ax.yaxis.set_major_formatter(LogFormatterMathtext(base=10))

        # lower bound must be > 0
        ymin = max(gene_df["normalized_count"][gene_df["normalized_count"] > 0].min() * 0.8, 1)

        plt.tight_layout()
        plt.savefig(output_dir / "deseq2" / "plots" / f"{gene}_expression.pdf")